# Project 4 — Customer Segmentation for Financial Services

**Objective:** Segment customers into distinct groups based on financial behavior to improve marketing strategies and service personalization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

## Step 1: Load Dataset

In [ ]:
df = pd.read_csv('4_customer_segment_data.csv')
print(f'Shape: {df.shape}')
df.head()

## Step 2: Data Preprocessing

In [ ]:
print('Missing values:')
print(df.isnull().sum())
df.dropna(inplace=True)

X = df.select_dtypes(include=[np.number])
print(f'Features: {X.columns.tolist()}')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('✅ Features scaled')

## Step 3: EDA — Correlation Heatmap & Boxplots

In [ ]:
print(df.describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[0], linewidths=0.5, square=True)
axes[0].set_title('Correlation Heatmap', fontsize=13)

df.plot(kind='box', ax=axes[1], vert=False, color='steelblue')
axes[1].set_title('Feature Distribution (Boxplots)', fontsize=13)

plt.tight_layout()
plt.show()

## Step 4: Determine Optimal K — Elbow Method

In [ ]:
inertias = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertias, marker='o', color='darkorange', linewidth=2)
plt.xticks(k_range)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method — Optimal k Selection', fontsize=13)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
print('→ Look for the "elbow" (bend) in the curve above to pick optimal k')

## Step 5: K-Means Clustering

In [ ]:
optimal_k = 4  # Adjust based on elbow curve above
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print(f'Cluster sizes (k={optimal_k}):')
print(df['Cluster'].value_counts().sort_index())

## Step 6: Visualize Clusters with PCA

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.tab10(np.linspace(0, 1, optimal_k))
sc = axes[0].scatter(X_pca[:, 0], X_pca[:, 1],
                      c=df['Cluster'], cmap='tab10', s=30, alpha=0.7)
axes[0].set_xlabel(f'PC1 ({var_exp[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({var_exp[1]*100:.1f}% variance)')
axes[0].set_title(f'PCA — KMeans Clusters (k={optimal_k})', fontsize=13)
plt.colorbar(sc, ax=axes[0], label='Cluster')
axes[0].grid(True, linestyle='--', alpha=0.4)

sizes = df['Cluster'].value_counts().sort_index()
axes[1].bar([f'Cluster {i}' for i in sizes.index], sizes.values,
            color=colors, edgecolor='black')
axes[1].set_title('Cluster Sizes', fontsize=13)
axes[1].set_ylabel('Number of Customers')
for i, v in enumerate(sizes.values):
    axes[1].text(i, v + 3, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## Step 7: Interpret Segments

In [ ]:
cluster_summary = df.groupby('Cluster').mean().round(2)
print('Mean feature values per cluster:')
print(cluster_summary)
cluster_summary.to_csv('customer_cluster_summary.csv')
print('\n✅ Summary saved → customer_cluster_summary.csv')

In [ ]:
n_features = len(X.columns)
fig, axes = plt.subplots(1, n_features, figsize=(4 * n_features, 4))
if n_features == 1: axes = [axes]

colors = plt.cm.tab10(np.linspace(0, 1, optimal_k))
for i, col in enumerate(X.columns):
    vals = cluster_summary[col]
    axes[i].bar([f'C{j}' for j in vals.index], vals.values, color=colors, edgecolor='black')
    axes[i].set_title(col, fontsize=11)
    axes[i].set_ylabel('Mean Value')

plt.suptitle('Mean Feature Values per Cluster', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Conclusion
- Customers were segmented using KMeans clustering with the optimal k determined by the **Elbow Method** (data-driven, not guessed).
- **EDA** (correlation heatmap + boxplots) was performed before clustering.
- **PCA** reduced dimensions to 2D for clear visual separation of clusters.
- **Cluster profiles** (mean feature values) were computed to interpret each segment.
- Segments can help tailor financial products and marketing strategies to each group.